In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import IngestionAgent

data_root = project_root / 'data' / 'v5.2'
agent = IngestionAgent(data_root=data_root)

print('Project root:', project_root)
print('Data root:', data_root)
print('Available datasets:', agent.available_datasets())
print('Feature subsets:', agent.feature_subset_names())

for dataset_name in ('train', 'validation', 'live'):
    summary = agent.summarize_dataset(
        dataset_name,
        feature_subset='medium',
        include_targets=dataset_name in {'train', 'validation'},
    )
    preview_columns = list(summary.columns[:8])
    print(f'{dataset_name}: rows={summary.row_count:,} columns={len(summary.columns)} path={summary.path.name}')
    print('  preview columns:', preview_columns)
    print('  preview schema:', {name: summary.schema[name] for name in preview_columns})

In [ ]:
from src.features import PurgedEraSplitter

splitter = PurgedEraSplitter(n_splits=5, purge_buffer=4)
train_lazy = agent.scan_dataset('train', include_metadata=True)
splits = splitter.split(train_lazy)

print('Fold | Training Eras | Validation Eras | Zero Leakage')
print('-----|---------------|-----------------|-------------')
for fold_number, (train_eras, validation_eras) in enumerate(splits, start=1):
    train_ordinals = [int(era) for era in train_eras]
    validation_ordinals = [int(era) for era in validation_eras]
    zero_leakage = all(abs(train_era - valid_era) > splitter.purge_buffer for train_era in train_ordinals for valid_era in validation_ordinals)
    print(f'{fold_number:>4} | {len(train_eras):>13} | {len(validation_eras):>15} | {zero_leakage}')

In [ ]:
import numpy as np
import polars as pl

from src.features import PurgedEraSplitter
from src.models import ModelOrchestrator

target_column = agent.get_target_names('train')[0]
selected_eras = (
    agent.scan_dataset('train', include_metadata=True)
    .select('era')
    .unique()
    .sort('era')
    .collect()
    .get_column('era')
    .to_list()[:30]
)
benchmark_frame = (
    agent.scan_dataset('train', feature_subset='small', include_metadata=True, include_targets=True)
    .filter(pl.col('era').is_in(selected_eras))
    .select(['id', 'era', target_column, *agent.get_feature_names('small')])
    .collect()
)
anchor_train_eras = selected_eras[:24]
anchor_val_eras = selected_eras[24:]
anchor_train = benchmark_frame.filter(pl.col('era').is_in(anchor_train_eras))
anchor_val = benchmark_frame.filter(pl.col('era').is_in(anchor_val_eras))
orchestrator = ModelOrchestrator(
    feature_names=agent.get_feature_names('small'),
    target_column=target_column,
    model_library='lightgbm',
    model_params={'n_estimators': 120, 'learning_rate': 0.05, 'num_leaves': 31, 'min_child_samples': 20},
    prefer_gpu=True,
    early_stopping_rounds=20,
)
anchor_result = orchestrator.train_anchor_fold(anchor_train, anchor_val)
cv_result = orchestrator.train_cross_validation(
    benchmark_frame,
    PurgedEraSplitter(n_splits=5, purge_buffer=1),
)
print(f'Anchor Loop Execution Time: {anchor_result.fit_seconds:.3f} s')
print(f'5-Fold Cross-Validation Execution Time: {cv_result.total_fit_seconds:.3f} s')
print('Anchor validation prediction preview:', np.round(anchor_result.validation_predictions[:5], 6).tolist())
oof_preview = cv_result.oof_predictions[np.isfinite(cv_result.oof_predictions)][:5]
print('OOF prediction preview:', np.round(oof_preview, 6).tolist())
print('Backends:', anchor_result.backend, cv_result.backend)